# Multi-Agent Personalized Travel Planner for Riyadh
## M4 — Single-Agent Baseline

This notebook implements the single-agent baseline using the same datasets and structured inputs defined in the research design.


In [ ]:
%pip install -q google-genai==1.30.0 gradio pandas numpy

In [ ]:
from pathlib import Path
import zipfile

DATA_DIR = Path("/content/riyadh_project/data")
required = ["activities.csv", "restaurants.csv", "hotels.csv"]

if not all((DATA_DIR / name).exists() for name in required):
    from google.colab import files

    print("Upload riyadh_m4_datasets.zip")
    uploaded = files.upload()

    if "riyadh_m4_datasets.zip" not in uploaded:
        raise FileNotFoundError("Required dataset ZIP not uploaded")

    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile("riyadh_m4_datasets.zip") as archive:
        expected = {f"data/{name}" for name in required}

        if not expected.issubset(set(archive.namelist())):
            raise ValueError("Dataset ZIP is incomplete")

        archive.extractall(
            DATA_DIR.parent,
            members=sorted(expected)
        )

print("All dataset files are ready!")


In [ ]:
import pandas as pd
import numpy as np
import json
import math
import time
from itertools import combinations
from pathlib import Path

DATA_DIR = Path("/content/riyadh_project/data")

activities = pd.read_csv(DATA_DIR / "activities.csv")
restaurants = pd.read_csv(DATA_DIR / "restaurants.csv")
hotels = pd.read_csv(DATA_DIR / "hotels.csv")

print("Activities:", activities.shape)
print("Restaurants:", restaurants.shape)
print("Hotels:", hotels.shape)

assert activities["place_id"].is_unique
assert restaurants["place_id"].is_unique
assert hotels["place_id"].is_unique


In [ ]:

# Official user input options from M2

INTEREST_OPTIONS = [
    "Shopping",
    "Entertainment",
    "Culture & History",
    "Nature & Outdoors",
    "Religious",
    "Sports",
    "Landmarks & Sightseeing"
]

BUDGET_OPTIONS = ["Low", "Medium", "High"]

# Verify that all selectable interests exist in the dataset
available_interests = set(
    activities["interest_group"].dropna().unique()
)

assert set(INTEREST_OPTIONS).issubset(available_interests)

# Verify budget levels
assert set(BUDGET_OPTIONS).issubset(
    set(restaurants["budget_level"].dropna().unique())
)

assert set(BUDGET_OPTIONS).issubset(
    set(hotels["budget_level"].dropna().unique())
)

print("User input options validated successfully!")

print("\nInterests:")
print(INTEREST_OPTIONS)

print("\nBudget levels:")
print(BUDGET_OPTIONS)


In [ ]:
def search_activities(interests, limit=20, exclude_ids=None):
    """
    Search activities matching user interests.
    Returns dataset-backed activity candidates.
    """

    if isinstance(interests, str):
        interests = [interests]

    if not interests or not set(interests).issubset(INTEREST_OPTIONS):
        raise ValueError("Invalid interests")

    if not isinstance(limit, int) or limit < 1:
        raise ValueError("limit must be a positive integer")

    exclude_ids = set(exclude_ids or [])

    # Filter by selected interests
    results = activities[
        activities["interest_group"].isin(interests)
    ].copy()

    # Exclude previously selected activities
    results = results[
        ~results["place_id"].isin(exclude_ids)
    ]

    # Prefer activities with more rating evidence
    results["_rating"] = pd.to_numeric(
        results["rating"], errors="coerce"
    ).fillna(-1)

    results["_rating_count"] = pd.to_numeric(
        results["number_of_ratings"], errors="coerce"
    ).fillna(0)

    results = results.sort_values(
        by=["_rating", "_rating_count", "place_id"],
        ascending=[False, False, True]
    )

    columns = [
        "place_id",
        "name",
        "interest_group",
        "rating",
        "number_of_ratings"
    ]

    return results[columns].head(limit).to_dict(
        orient="records"
    )

In [ ]:
def search_restaurants(budget_level, limit=20, exclude_ids=None):
    """
    Search restaurants by the user's budget level.
    """

    if budget_level not in BUDGET_OPTIONS:
        raise ValueError("Invalid budget level")

    if not isinstance(limit, int) or limit < 1:
        raise ValueError("limit must be a positive integer")

    exclude_ids = set(exclude_ids or [])

    results = restaurants[
        restaurants["budget_level"] == budget_level
    ].copy()

    results = results[
        ~results["place_id"].isin(exclude_ids)
    ]

    results["_rating"] = pd.to_numeric(
        results["rating"], errors="coerce"
    ).fillna(-1)

    results["_rating_signals"] = pd.to_numeric(
        results["ratingSignals"], errors="coerce"
    ).fillna(0)

    results = results.sort_values(
        by=["_rating", "_rating_signals", "place_id"],
        ascending=[False, False, True]
    )

    columns = [
        "place_id",
        "name",
        "budget_level",
        "rating",
        "ratingSignals"
    ]

    return results[columns].head(limit).to_dict(
        orient="records"
    )

In [ ]:
def search_hotels(budget_level, limit=10):
    """
    Search hotels matching the user's budget level.
    """

    if budget_level not in BUDGET_OPTIONS:
        raise ValueError("Invalid budget level")

    if not isinstance(limit, int) or limit < 1:
        raise ValueError("limit must be a positive integer")

    results = hotels[
        hotels["budget_level"] == budget_level
    ].copy()

    results["_rating"] = pd.to_numeric(
        results["rating"], errors="coerce"
    ).fillna(-1)

    results["_count"] = pd.to_numeric(
        results["count"], errors="coerce"
    ).fillna(0)

    results = results.sort_values(
        by=["_rating", "_count", "place_id"],
        ascending=[False, False, True]
    )

    columns = [
        "place_id",
        "hotel_name",
        "budget_level",
        "median_price",
        "rating",
        "count"
    ]

    return results[columns].head(limit).to_dict(
        orient="records"
    )

In [ ]:
# Fixed: default limit raised from 10 to 30 — a 5-day trip needs 10 unique activities,
# so the model needs a larger pool to choose the best matches from
def find_activities(interests: list[str], limit: int = 30) -> list[dict]:
    """
    Search Riyadh activities matching the user's interests.

    Args:
        interests: Selected interest groups.
        limit: Maximum number of activities to return.

    Returns:
        A list of matching activities from the dataset.
    """
    return search_activities(
        interests=interests,
        limit=limit
    )

In [ ]:
def find_restaurants(
    budget_level: str,
    limit: int = 10
) -> list[dict]:
    """
    Search Riyadh restaurants by budget level.

    Args:
        budget_level: Low, Medium, or High.
        limit: Maximum number of restaurants.

    Returns:
        Matching restaurants from the dataset.
    """
    return search_restaurants(
        budget_level=budget_level,
        limit=limit
    )

In [ ]:
def find_hotels(
    budget_level: str,
    limit: int = 5
) -> list[dict]:
    """
    Search Riyadh hotels matching the user's budget.

    Args:
        budget_level: Low, Medium, or High.
        limit: Maximum number of hotels to return.

    Returns:
        Matching hotels from the dataset.
    """
    return search_hotels(
        budget_level=budget_level,
        limit=limit
    )

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types, errors

TEST_MODEL = "gemini-3.5-flash-lite"

api_key = userdata.get("GEMINI_API_KEY")
assert api_key, "Add GEMINI_API_KEY to Colab Secrets."

client = genai.Client(api_key=api_key)

print("Gemini client initialized.")


In [ ]:
def validate_itinerary(itinerary, trip_duration):
    errors = []

    hotel_ids = set(hotels["place_id"])
    activity_ids = set(activities["place_id"])
    restaurant_ids = set(restaurants["place_id"])

    if not isinstance(itinerary, dict):
        return False, ["Itinerary must be a dictionary"]

    hotel_id = itinerary.get("hotel_id")

    if not isinstance(hotel_id, str) or hotel_id not in hotel_ids:
        errors.append("Invalid hotel ID")

    days = itinerary.get("days")

    if not isinstance(days, list):
        return False, errors + ["Days must be a list"]

    if len(days) != trip_duration:
        errors.append("Incorrect number of days")

    all_activities = []

    for index, day in enumerate(days, start=1):
        if not isinstance(day, dict):
            errors.append(f"Day {index}: invalid structure")
            continue

        if day.get("day") != index:
            errors.append(f"Day {index}: incorrect day number")

        day_activities = day.get("activities")
        day_restaurants = day.get("restaurants")

        if not isinstance(day_activities, list):
            errors.append(f"Day {index}: activities must be a list")
        else:
            if len(day_activities) != 2:
                errors.append(f"Day {index}: must have 2 activities")

            for pid in day_activities:
                if not isinstance(pid, str) or pid not in activity_ids:
                    errors.append(f"Day {index}: invalid activity ID")

            all_activities.extend(day_activities)

        if not isinstance(day_restaurants, list):
            errors.append(f"Day {index}: restaurants must be a list")
        else:
            if len(day_restaurants) != 2:
                errors.append(f"Day {index}: must have 2 restaurants")

            for pid in day_restaurants:
                if not isinstance(pid, str) or pid not in restaurant_ids:
                    errors.append(f"Day {index}: invalid restaurant ID")

    if len(all_activities) != len(set(all_activities)):
        errors.append("Repeated activities found")

    return len(errors) == 0, errors

In [ ]:
import math

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1, lon1, lat2, lon2 = map(
        math.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    return 2 * R * math.asin(math.sqrt(a))

def get_place_coordinates(place_id):
    if place_id.startswith("ACT_"):
        row = activities.loc[
            activities["place_id"] == place_id
        ].iloc[0]
        return float(row["latitude"]), float(row["longitude"])

    if place_id.startswith("RES_"):
        row = restaurants.loc[
            restaurants["place_id"] == place_id
        ].iloc[0]
        return float(row["lat"]), float(row["lng"])

    if place_id.startswith("HOT_"):
        row = hotels.loc[
            hotels["place_id"] == place_id
        ].iloc[0]

        if row[["latitude", "longitude"]].isna().any():
            return None

        return float(row["latitude"]), float(row["longitude"])

    raise ValueError(f"Unknown place ID: {place_id}")

In [ ]:
from itertools import combinations

def evaluate_geography(itinerary):
    daily_results = []

    for day in itinerary["days"]:
        place_ids = day["activities"] + day["restaurants"]

        coordinates = [
            get_place_coordinates(pid)
            for pid in place_ids
        ]

        valid_coordinates = all(
            coord is not None
            and all(math.isfinite(v) for v in coord)
            and 24.3 <= coord[0] <= 25.2
            and 46.2 <= coord[1] <= 47.2
            for coord in coordinates
        )

        if valid_coordinates:
            distances = [
                haversine_km(*coord1, *coord2)
                for coord1, coord2 in combinations(coordinates, 2)
            ]

            mean_distance = sum(distances) / len(distances)
            max_distance = max(distances)
        else:
            mean_distance = None
            max_distance = None

        daily_results.append({
            "day": day["day"],
            "valid_coordinates": valid_coordinates,
            "mean_distance_km": mean_distance,
            "max_distance_km": max_distance
        })

    hotel_coordinates = get_place_coordinates(
        itinerary["hotel_id"]
    )

    return {
        "daily_results": daily_results,
        "hotel_coordinates_available": (
            hotel_coordinates is not None
        ),
        "all_daily_coordinates_valid": all(
            day["valid_coordinates"]
            for day in daily_results
        )
    }

In [ ]:
def evaluate_itinerary(itinerary, trip_duration, interests, budget_level):
    structure_valid, errors = validate_itinerary(
        itinerary, trip_duration
    )

    if not structure_valid:
        return {
            "structure_valid": False,
            "validation_errors": errors,
            "activity_preference_match_pct": None,
            "restaurant_budget_match_pct": None,
            "hotel_budget_match": None,
            "geography": None
        }

    activity_lookup = activities.set_index("place_id")
    restaurant_lookup = restaurants.set_index("place_id")
    hotel_lookup = hotels.set_index("place_id")

    activity_ids = [
        pid for day in itinerary["days"]
        for pid in day["activities"]
    ]

    restaurant_ids = [
        pid for day in itinerary["days"]
        for pid in day["restaurants"]
    ]

    activity_matches = [
        activity_lookup.loc[pid, "interest_group"] in interests
        for pid in activity_ids
    ]

    restaurant_matches = [
        restaurant_lookup.loc[pid, "budget_level"] == budget_level
        for pid in restaurant_ids
    ]

    hotel_match = (
        hotel_lookup.loc[itinerary["hotel_id"], "budget_level"]
        == budget_level
    )

    geography = evaluate_geography(itinerary)

    return {
        "structure_valid": True,
        "validation_errors": [],
        "activity_preference_match_pct": round(
            100 * sum(activity_matches) / len(activity_matches), 2
        ),
        "restaurant_budget_match_pct": round(
            100 * sum(restaurant_matches) / len(restaurant_matches), 2
        ),
        "hotel_budget_match": bool(hotel_match),
        "geography": geography
    }

In [ ]:
def run_baseline(trip_duration, interests, budget_level):
    # Step 1: Generate itinerary using the LLM agent
    generation = generate_baseline_itinerary(
        trip_duration=trip_duration,
        interests=interests,
        budget_level=budget_level
    )

    # Step 2: Evaluate the generated itinerary
    evaluation = evaluate_itinerary(
        itinerary=generation["itinerary"],
        trip_duration=trip_duration,
        interests=interests,
        budget_level=budget_level
    )

    # Step 3: Return complete results
    return {
        "user_input": {
            "trip_duration": trip_duration,
            "interests": interests,
            "budget_level": budget_level
        },
        "model": generation["model"],
        "itinerary": generation["itinerary"],
        "tool_calls": generation["tool_calls"],
        "evaluation": evaluation
    }

In [ ]:
import time

from google.genai import errors

def call_gemini_with_retry(prompt, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        try:
            response = client.models.generate_content(
                model=TEST_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    tools=[
                        find_activities,
                        find_restaurants,
                        find_hotels
                    ],
                    temperature=0
                )
            )

            return response

        except errors.ServerError as e:
            if e.code not in [500, 502, 503, 504]:
                raise

            print(
                f"Temporary API error {e.code} "
                f"(attempt {attempt}/{max_attempts})"
            )

            if attempt == max_attempts:
                raise

            time.sleep(5 * attempt)

        # Fixed: catch 429 rate-limit errors (classified as ClientError in google-genai SDK)
        except errors.ClientError as e:
            if e.code != 429:
                raise

            print(
                f"Rate limited (429) "
                f"(attempt {attempt}/{max_attempts})"
            )

            if attempt == max_attempts:
                raise

            time.sleep(5 * attempt)

In [ ]:
def generate_baseline_itinerary(
    trip_duration,
    interests,
    budget_level
):
    # Validate user inputs
    allowed_interests = [
        "Shopping",
        "Entertainment",
        "Culture & History",
        "Nature & Outdoors",
        "Religious",
        "Sports",
        "Landmarks & Sightseeing"
    ]

    if type(trip_duration) is not int or trip_duration < 1:
        raise ValueError("Trip duration must be a positive integer")

    if (
        not isinstance(interests, list)
        or not interests
        or any(i not in allowed_interests for i in interests)
    ):
        raise ValueError("Invalid interests")

    if budget_level not in ["Low", "Medium", "High"]:
        raise ValueError("Invalid budget level")

    # Same standardized prompt as before
    prompt = (
        "You are a single-agent Riyadh travel planner. "
        f"Plan a {trip_duration}-day trip. "
        f"User interests: {json.dumps(interests)}. "
        f"Budget level: {budget_level}. "
        "Use all three tools: find_activities, "
        "find_restaurants, and find_hotels. "
        "Select exactly one hotel for the whole trip. "
        "Select exactly two activities and two restaurants per day. "
        "Do not repeat activities. "
        "Only select place_ids returned by the tools. "
        "Use the requested budget level for hotels and restaurants. "
        "Prioritize activities matching the user's interests. "
        "Return ONLY valid JSON with keys hotel_id and days. "
        "Each day must contain day, activities, and restaurants. "
        "Activities and restaurants must each contain exactly two IDs. "
        "Do not include markdown or explanations."
    )

    # Updated: API call with bounded retries
    response = call_gemini_with_retry(prompt)

    # Fixed: strip markdown code fences (```json...```) that LLMs sometimes wrap around JSON output
    raw_text = response.text.strip()
    if raw_text.startswith("```"):
        # Remove opening fence (```json or ```) and closing fence (```)
        raw_text = raw_text.split("\n", 1)[-1]  # remove first line
        raw_text = raw_text.rsplit("```", 1)[0]  # remove last ```
        raw_text = raw_text.strip()

    itinerary = json.loads(raw_text)

    # Validate itinerary structure and IDs
    is_valid, validation_errors = validate_itinerary(
        itinerary,
        trip_duration
    )

    # Record actual tool calls
    tool_calls = []

    for message in (
        response.automatic_function_calling_history or []
    ):
        for part in message.parts:
            if part.function_call:
                tool_calls.append(part.function_call.name)

    return {
        "itinerary": itinerary,
        "is_valid": is_valid,
        "validation_errors": validation_errors,
        "tool_calls": tool_calls,
        "model": TEST_MODEL
    }

In [ ]:

import gradio as gr

def get_place_name(df, place_id, name_col):
    match = df.loc[df["place_id"] == place_id]
    if match.empty:
        return f"Unknown place ({place_id})"
    return str(match.iloc[0][name_col])

def generate_trip_pretty(days, interests, budget):
    if not interests:
        return "Please select at least one interest."

    try:
        result = run_baseline(
            trip_duration=int(days),
            interests=interests,
            budget_level=budget
        )

        itinerary = result["itinerary"]
        evaluation = result["evaluation"]

        lines = [
            "# Your Riyadh Travel Itinerary",
            f"**Duration:** {days} days",
            f"**Interests:** {', '.join(interests)}",
            f"**Budget:** {budget}",
            ""
        ]

        hotel_id = itinerary["hotel_id"]
        hotel_name = get_place_name(
            hotels, hotel_id, "hotel_name"
        )
        lines.append(f"## Hotel: {hotel_name}")
        lines.append(f"`{hotel_id}`")
        lines.append("")

        for day in itinerary["days"]:
            lines.append(f"## Day {day['day']}")
            lines.append("**Activities**")

            for activity_id in day["activities"]:
                name = get_place_name(
                    activities, activity_id, "name"
                )
                lines.append(f"- {name} (`{activity_id}`)")

            lines.append("")
            lines.append("**Restaurants**")

            for restaurant_id in day["restaurants"]:
                name = get_place_name(
                    restaurants, restaurant_id, "name"
                )
                lines.append(f"- {name} (`{restaurant_id}`)")

            lines.append("")

        # --- Fixed: handle case where structure is invalid (geography=None) ---
        lines.extend([
            "---",
            "## Evaluation Results",
            f"- Structure Valid: {evaluation['structure_valid']}",
            f"- Activity Preference Match: {evaluation['activity_preference_match_pct']}%",
            f"- Restaurant Budget Match: {evaluation['restaurant_budget_match_pct']}%",
            f"- Hotel Budget Match: {evaluation['hotel_budget_match']}",
        ])

        # Only show geography if evaluation succeeded (not None)
        if evaluation['geography']:
            lines.append(
                f"- All Daily Coordinates Valid: "
                f"{evaluation['geography']['all_daily_coordinates_valid']}"
            )
        else:
            lines.append("- Geography: N/A (structure invalid)")

        lines.append(f"- Tools Used: {', '.join(result['tool_calls'])}")

        return "\n".join(lines)

    except Exception as e:
        return f"**Generation failed:** {type(e).__name__}: {e}"

with gr.Blocks(title="Riyadh Travel Planner") as pretty_demo:
    gr.Markdown("# Riyadh Travel Planner")
    gr.Markdown(
        "Single-Agent AI Baseline | Personalized Riyadh Trips"
    )

    days = gr.Dropdown(
        choices=[1, 2, 3, 4, 5],
        value=2,
        label="Trip Duration (Days)"
    )

    interests = gr.CheckboxGroup(
        choices=[
            "Shopping", "Entertainment",
            "Culture & History", "Nature & Outdoors",
            "Religious", "Sports",
            "Landmarks & Sightseeing"
        ],
        value=["Entertainment"],
        label="Interests / Preferences"
    )

    budget = gr.Radio(
        choices=["Low", "Medium", "High"],
        value="Medium",
        label="Budget Level"
    )

    generate_button = gr.Button(
        "Generate Itinerary",
        variant="primary"
    )

    output = gr.Markdown(label="Your Travel Plan")

    generate_button.click(
        fn=generate_trip_pretty,
        inputs=[days, interests, budget],
        outputs=output
    )

pretty_demo.launch(share=False)